# Cyberbullying Classification Using Bi-LSTM
## Project Progress Notebook

**Course:** Artificial Neural Networks and Deep Learning Lab Project  
**Author / Student:** Zia Malik  
**Current Stage:** Stage 1 — Dataset Inspection, Duplicate & Conflict Analysis, Data Quality Policy Selection

---
## Project Repository

**GitHub Repository:** https://github.com/ZiaMalik-wq/cyberbullying-bilstm

The repository contains the project's complete source code, documentation, experiments, and version history.


---
## 1. Project Overview

### 1.1 Objective
Cyberbullying on social media platforms presents serious psychological and social challenges. While traditional binary detection systems classify text merely as "cyberbullying" or "not cyberbullying", real-world moderation requires **fine-grained multiclass classification** to identify the specific nature of abuse. This project develops a deep-learning-based classification system to categorize social media posts into **six distinct categories**:
1. `age` — Age-based cyberbullying (e.g., ageist slurs, harassment targeting elderly or youth).
2. `ethnicity` — Harassment targeting race, ethnic background, or nationality.
3. `gender` — Sexist, misogynistic, or gender-based harassment.
4. `religion` — Religious bias, bigotry, and sectarian abuse.
5. `other_cyberbullying` — Unstructured or general bullying not fitting specific demographic types.
6. `not_cyberbullying` — Benign, neutral, or non-abusive discourse.

### 1.2 Research Questions
* **Primary Research Question:**  
  *How effectively can a Bidirectional Long Short-Term Memory (Bi-LSTM) network trained entirely from scratch classify different types of cyberbullying from social-media text, and how does it compare with alternative recurrent neural architectures?*
* **Secondary Research Questions:**
  1. How does a Simple RNN baseline compare against LSTM, GRU, and Bi-LSTM under controlled conditions?
  2. How do hyperparameter variations (sequence length, embedding dimensions, recurrent units, dropout) influence classification accuracy and macro F1?
  3. Which cyberbullying categories exhibit the highest cross-class confusion, and what linguistic patterns characterize misclassifications?

### 1.3 Core Deep Learning Constraints (Strict University Lab Policy)
* **Zero Pretrained Representations:**  
  The use of pretrained language models or pretrained static embeddings (**BERT, RoBERTa, DistilBERT, GPT, Word2Vec, GloVe, FastText, sentence-transformers**) is **strictly forbidden**.
* **Trained From Scratch:**  
  All token embeddings and network parameters must be randomly initialized and trained entirely from the project's training data.
* **Strict Leakage Prevention:**  
  The vocabulary must be derived solely from the training split. The test set is strictly sequestered for final, un-tuned evaluation.


---
## 2. Environment and Libraries

This notebook is designed to execute seamlessly in both:
1. **Google Colab / Kaggle Notebook** (cloud GPU runtime)
2. **Local Development Environment** (workstation, Python virtual environment)

### 2.1 Colab Workflow & Repository Synchronization
When running in Google Colab, the cell below automates the environment setup:
1. Checks if `/content/cyberbullying-bilstm` exists.
   - **No:** Clones the repository (`git clone`).
   - **Yes:** Pulls latest committed code (`git pull origin main`).
2. Sets `PROJECT_ROOT` to `/content/cyberbullying-bilstm` and changes working directory.
3. Adds `PROJECT_ROOT` to `sys.path` so that reusable logic in `src/` can be imported directly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Repository configuration
REPO_URL = "https://github.com/ZiaMalik-wq/cyberbullying-bilstm.git"
REPO_NAME = "cyberbullying-bilstm"

# 1. Environment Detection and Repository Synchronization
try:
    import google.colab
    IN_COLAB = True
    colab_dir = Path(f"/content/{REPO_NAME}")
    
    if not colab_dir.exists():
        print(f"Directory {colab_dir} not found. Cloning repository from GitHub...")
        subprocess.run(["git", "clone", REPO_URL, str(colab_dir)], check=True)
        print("Repository cloned successfully.")
    else:
        print(f"Directory {colab_dir} exists. Pulling latest committed changes from GitHub...")
        subprocess.run(["git", "-C", str(colab_dir), "pull", "origin", "main"], check=True)
        print("Repository updated with latest commits.")
        
    PROJECT_ROOT = colab_dir
    os.chdir(PROJECT_ROOT)
    print(f"Running in Google Colab. Working directory set to: {PROJECT_ROOT}")

except ImportError:
    IN_COLAB = False
    current_dir = Path.cwd().resolve()
    if current_dir.name == "notebooks":
        PROJECT_ROOT = current_dir.parent
    else:
        PROJECT_ROOT = current_dir
    print(f"Running in local environment. Project root: {PROJECT_ROOT}")

# 2. Add PROJECT_ROOT to sys.path to enable importing modules from src/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Verified sys.path includes: {PROJECT_ROOT}")


In [ ]:
# 2.2 Core Scientific Computing & Visualization Libraries
import json
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTTING = True
    plt.rcParams["figure.figsize"] = (9, 5)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
except ImportError:
    HAS_PLOTTING = False
    print("Note: matplotlib/seaborn not found; using textual summaries.")

# Global reproducibility seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Environment initialized successfully.")
print(f"Pandas version : {pd.__version__}")
print(f"NumPy version  : {np.__version__}")


---
## 3. Dataset

### 3.1 Dataset Source and Provenance
* **Dataset:** Cyberbullying Classification Dataset
* **Platform:** Kaggle (<https://www.kaggle.com/datasets/andrewmvd/cyberbullying-classification>)
* **Domain:** Social media posts (Twitter/X) annotated across 6 classes.
* **File Location:** `data/raw/cyberbullying_tweets.csv`
* **Integrity Constraint:** The raw dataset is strictly **read-only** and must never be altered or overwritten.

### 3.2 Dataset Schema
The raw dataset consists of two columns:
1. `tweet_text` (*string*): The raw user-generated social media post.
2. `cyberbullying_type` (*string*): The assigned ground-truth cyberbullying label.


In [ ]:
# 3.1 Check Raw Dataset (Exists -> Use it; Missing -> Ask to upload)
import shutil
from pathlib import Path

raw_data_dir = PROJECT_ROOT / "data" / "raw"
raw_data_path = raw_data_dir / "cyberbullying_tweets.csv"

if not raw_data_path.exists():
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    print("=" * 70)
    print(f"Raw dataset NOT found at: {raw_data_path}")
    print("Because data/raw/ is excluded from Git tracking (.gitignore),")
    print("the raw dataset must be uploaded when running in a fresh environment.")
    print("=" * 70)
    
    if IN_COLAB:
        print("\nPlease upload 'cyberbullying_tweets.csv' using the file picker below:")
        from google.colab import files
        uploaded = files.upload()
        for filename in uploaded.keys():
            if filename.endswith(".csv"):
                shutil.move(filename, str(raw_data_path))
                print(f"\n[OK] Uploaded file saved to: {raw_data_path}")
                break
    else:
        raise FileNotFoundError(
            f"Raw dataset file not found at {raw_data_path}. "
            "Please download 'cyberbullying_tweets.csv' and place it in 'data/raw/'."
        )
else:
    print(f"[OK] Raw dataset found at: {raw_data_path}")


In [ ]:
# 3.2 Load Raw Dataset using Reusable Module from src/
from src.data_cleaning import load_raw_csv, DEFAULT_TEXT_COL, DEFAULT_LABEL_COL

# Load raw dataset using reusable loader from src/
df_raw = load_raw_csv(raw_data_path)

print(f"Loaded raw dataset from: {raw_data_path.relative_to(PROJECT_ROOT)}")
print(f"Total samples (rows): {len(df_raw):,}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()


---
## 4. Dataset Inspection

Before designing preprocessing, splitting, or neural architectures, a comprehensive audit of data hygiene, null values, class balance, and duplicate entries is essential.


In [ ]:
# 4.1 Basic Shape and Missing Value Audit
print("=" * 60)
print("4.1 DATASET HYGIENE AUDIT")
print("=" * 60)
print(f"DataFrame dimensions : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Memory usage         : {df_raw.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
print("\nMissing values per column:")
print(df_raw.isnull().sum())

assert df_raw[DEFAULT_TEXT_COL].isnull().sum() == 0, "Missing values found in text column!"
assert df_raw[DEFAULT_LABEL_COL].isnull().sum() == 0, "Missing values found in label column!"
print("\nValidation Result: Zero null or missing values detected.")


In [ ]:
# 4.2 Raw Class Distribution
class_counts = df_raw[DEFAULT_LABEL_COL].value_counts()
class_pcts = df_raw[DEFAULT_LABEL_COL].value_counts(normalize=True) * 100

dist_table = pd.DataFrame({
    "Sample Count": class_counts,
    "Percentage (%)": class_pcts.round(2)
})
print("=" * 60)
print("4.2 RAW CLASS DISTRIBUTION")
print("=" * 60)
print(dist_table)

if HAS_PLOTTING:
    plt.figure(figsize=(9, 4.5))
    palette = sns.color_palette("muted", len(class_counts))
    bars = plt.bar(class_counts.index, class_counts.values, color=palette, edgecolor="black", alpha=0.85)
    plt.title("Baseline Class Distribution (Raw Dataset)", fontsize=13, fontweight="bold", pad=12)
    plt.xlabel("Cyberbullying Category", fontsize=11)
    plt.ylabel("Number of Samples", fontsize=11)
    plt.xticks(rotation=25, ha="right")
    plt.ylim(0, class_counts.max() * 1.15)

    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 100, f"{yval:,}\n({yval/len(df_raw)*100:.1f}%)", 
                 ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()


### 4.3 Duplicate and Conflicting-Label Analysis

A critical challenge discovered in the raw dataset is the presence of **identical tweet texts** appearing with **contradictory labels**.

To analyze the duplicate landscape quantitatively:
1. **Exact Duplicate Rows:** Rows where *both* `tweet_text` and `cyberbullying_type` are completely identical.
2. **Consistent Duplicate Groups:** Unique texts appearing multiple times under the *same* label (redundant repetitions).
3. **Conflicting-Label Groups:** Unique texts appearing multiple times under *different* labels (ambiguous/contradictory supervision).


In [ ]:
# Quantitative Duplicate & Conflict Audit
text_col = DEFAULT_TEXT_COL
label_col = DEFAULT_LABEL_COL

n_total = len(df_raw)
n_unique_texts = df_raw[text_col].nunique()
text_counts = df_raw[text_col].value_counts()
repeated_texts = text_counts[text_counts > 1].index

# Group texts by unique label count
label_nunique_per_text = df_raw.groupby(text_col)[label_col].nunique()
conflicting_texts = label_nunique_per_text[label_nunique_per_text > 1].index
consistent_dup_texts = text_counts[(text_counts > 1) & (~text_counts.index.isin(conflicting_texts))].index

n_conflicting_rows = df_raw[df_raw[text_col].isin(conflicting_texts)].shape[0]
n_consistent_rows = df_raw[df_raw[text_col].isin(consistent_dup_texts)].shape[0]

print("=" * 60)
print("4.3 DUPLICATE AND CONFLICTING LABEL AUDIT")
print("=" * 60)
print(f"Total Rows in Raw Dataset                     : {n_total:,}")
print(f"Unique Tweet Texts                            : {n_unique_texts:,}")
print(f"Texts appearing exactly once                  : {(text_counts == 1).sum():,}")
print(f"Texts appearing more than once                : {len(repeated_texts):,}")
print("-" * 60)
print(f"Consistent Duplicate Groups (same label)      : {len(consistent_dup_texts):,} texts ({n_consistent_rows} rows)")
print(f"Conflicting-Label Groups (different labels)   : {len(conflicting_texts):,} texts ({n_conflicting_rows:,} rows)")
print("=" * 60)

# Check frequency of appearances among conflicting texts
conflict_freq = df_raw[df_raw[text_col].isin(conflicting_texts)][text_col].value_counts()
print(f"Conflicting texts appearing exactly 2 times   : {(conflict_freq == 2).sum():,} (100.0%)")
print(f"Conflicting texts appearing 3+ times          : {(conflict_freq >= 3).sum():,} (0.0%)")


In [ ]:
# Breakdown of Conflicting Label Pairs
conflicting_df = df_raw[df_raw[text_col].isin(conflicting_texts)]
conflict_pairs = (
    conflicting_df.groupby(text_col)[label_col]
    .apply(lambda s: " + ".join(sorted(s.unique())))
    .value_counts()
)

conflict_pair_df = pd.DataFrame({
    "Conflicting Label Pair": conflict_pairs.index,
    "Unique Texts": conflict_pairs.values,
    "Rows Involved": conflict_pairs.values * 2,
    "% of Conflicts": (conflict_pairs.values / len(conflicting_texts) * 100).round(2)
})
print("Top Conflicting Label Combinations:")
print(conflict_pair_df.to_string(index=False))


---
## 5. Data Quality Policy

### 5.1 The Decision: Policy D
Based on the quantitative findings in `docs/DUPLICATE_ANALYSIS.md` and `docs/DUPLICATE_POLICY_IMPACT.md`, the project selected **Policy D** as formal policy (`docs/DATA_QUALITY_POLICY.md`):

1. **Remove Conflicting Texts:**  
   Every row belonging to the 1,639 text groups with contradictory labels (covering 3,278 rows) is completely excluded.
2. **Deduplicate Consistent Duplicates:**  
   For the 36 text groups that appear multiple times with the exact same label, keep exactly one copy and discard the 36 redundant copies.
3. **Strict Raw Preservation:**  
   The raw dataset `data/raw/cyberbullying_tweets.csv` remains strictly untouched. The cleaned dataset is exported to `data/processed/cyberbullying_clean.csv`.

### 5.2 Rationale: Why Majority Voting and Policy B Were Rejected
* **Majority Voting is Mathematically Impossible:**  
  100% of conflicting texts in this dataset appear exactly twice (1 vote for Label A vs 1 vote for Label B). There is not a single 3+ occurrence case. Every conflict is a strict 1-vs-1 tie.
* **Policy B (First-Seen Heuristic) is Unscientific:**  
  Retaining the first occurrence arbitrarily picks whichever label happened to come first in the raw CSV ordering. Raw row ordering is non-semantic noise; assigning ground-truth based on file order injects arbitrary label noise into both training and evaluation.
* **Data Leakage & Ambiguity Elimination:**  
  Removing conflicting texts guarantees that the exact same text cannot exist in both train and test splits under opposing supervisory targets.


In [ ]:
from src.data_cleaning import apply_policy_d

# Apply Policy D using the validated implementation from src/
df_clean = apply_policy_d(df_raw, text_col=DEFAULT_TEXT_COL, label_col=DEFAULT_LABEL_COL)

# Rigorous post-cleaning verification
assert len(df_clean) == 44378, f"Expected 44,378 rows, got {len(df_clean)}"
assert df_clean[DEFAULT_TEXT_COL].nunique() == 44378, "Duplicate texts detected in cleaned dataset!"
assert (df_clean.groupby(DEFAULT_TEXT_COL)[DEFAULT_LABEL_COL].nunique() > 1).sum() == 0, "Conflicts remain!"

print("=" * 60)
print("5.1 POLICY D CLEANING VERIFICATION")
print("=" * 60)
print(f"Raw rows               : {len(df_raw):,}")
print(f"Rows removed           : {len(df_raw) - len(df_clean):,} (-{(len(df_raw)-len(df_clean))/len(df_raw)*100:.2f}%)")
print(f"Cleaned rows retained  : {len(df_clean):,} ({(len(df_clean)/len(df_raw))*100:.2f}%)")
print(f"Unique texts remaining : {df_clean[DEFAULT_TEXT_COL].nunique():,}")
print(f"Duplicates remaining   : 0")
print(f"Conflicts remaining    : 0")
print("Verification Status    : ALL INTEGRITY CHECKS PASSED [OK]")


In [ ]:
# 5.2 Before vs After Policy D Class Distribution Comparison
clean_counts = df_clean[DEFAULT_LABEL_COL].value_counts()
clean_pcts = df_clean[DEFAULT_LABEL_COL].value_counts(normalize=True) * 100

comparison_df = pd.DataFrame({
    "Raw Count": class_counts,
    "Raw %": class_pcts.round(2),
    "Clean Count": clean_counts,
    "Clean %": clean_pcts.round(2),
    "Delta Rows": clean_counts - class_counts,
    "Change (%)": ((clean_counts - class_counts) / class_counts * 100).round(2)
}).sort_values("Clean Count", ascending=False)

print("=" * 60)
print("5.2 CLASS DISTRIBUTION: RAW VS CLEANED (POLICY D)")
print("=" * 60)
print(comparison_df.to_string())

if HAS_PLOTTING:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(comparison_df))
    width = 0.35

    ax.bar(x - width/2, comparison_df["Raw Count"], width, label="Raw Dataset", color="#5C6BC0", alpha=0.9, edgecolor="black")
    ax.bar(x + width/2, comparison_df["Clean Count"], width, label="Cleaned (Policy D)", color="#26A69A", alpha=0.9, edgecolor="black")

    ax.set_title("Class Distribution Comparison: Raw vs Cleaned (Policy D)", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Cyberbullying Category", fontsize=11)
    ax.set_ylabel("Sample Count", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_df.index, rotation=25, ha="right")
    ax.legend(frameon=True, facecolor="white", loc="upper right")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()


---
## 6. Train / Validation / Test Split
> **Status: Planned / Not yet executed — Scheduled for Stage 2**
>
> *In accordance with project rules, the dataset split is planned here but NOT executed until Stage 2.*

### 6.1 Splitting Methodology
* **Split Ratios:** 70% Train, 15% Validation, 15% Test.
  * Train (~31,064 samples): Exclusively used for model parameter fitting and `TextVectorization` vocabulary learning.
  * Validation (~6,657 samples): Used for hyperparameter tuning, validation loss monitoring, and Early Stopping.
  * Test (~6,657 samples): Strictly isolated until final model evaluation.
* **Stratified Sampling:** `scikit-learn`'s `train_test_split` with `stratify=y` to preserve the exact class proportion across all three partitions.
* **Reproducibility:** Seed fixed to `RANDOM_SEED = 42`.
* **Zero Leakage Invariant:** No vocabulary adaptation, sequence statistics, or hyperparameter decisions will be influenced by the test split.


In [ ]:
# Planned configuration for Stage 2 Split (DO NOT EXECUTE SPLIT YET)
SPLIT_CONFIG = {
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "stratify": True,
    "random_seed": RANDOM_SEED,
    "status": "Pending execution in Stage 2"
}

print("Train / Val / Test Split Specification:")
print(json.dumps(SPLIT_CONFIG, indent=2))
print("\n[NOTE] Splitting execution deferred to Stage 2 implementation.")


---
## 7. Text Preprocessing
> **Status: Planned / Not yet executed**

### 7.1 Preprocessing Strategy for Social Media Text
In cyberbullying detection, aggressive preprocessing (such as blind removal of punctuation, capitalization, or stop words) can eliminate critical abusive signals (e.g., ALL-CAPS screaming, targeted exclamation, emojis).

The planned preprocessing pipeline will carefully evaluate:
- URL removal/tokenization (`https://...` -> `<URL>`).
- User mention handling (`@user` -> `<USER>`).
- Hashtag segmentation (preserving sentiment while standardizing syntax).
- Contraction expansion and character repetition normalization (e.g., "noooo" -> "no").
- Lowercasing (standardized text normalization).


---
## 8. Text Vectorization
> **Status: Planned / Not yet executed**

### 8.1 Vectorization Pipeline Design
* **Mechanism:** TensorFlow / Keras `TextVectorization` layer.
* **Strict Anti-Leakage Rule:** The vocabulary will be adapted **strictly on the Training Set only** (`vectorizer.adapt(train_texts)`). Neither validation nor test texts will be observed during vocabulary building.
* **Key Hyperparameters:**
  * `max_tokens` (vocabulary size): Default 10,000 to 20,000.
  * `output_sequence_length`: Default 100 to 128 tokens.
  * `standardize`: Custom standardization function preserving essential linguistic markers.
  * `output_mode`: `'int'` (token sequence indices with zero-padding and OOV token handling).


---
## 9. Baseline Model
> **Status: Planned / Not yet executed**

### 9.1 Baseline Architecture
Prior to training recurrent networks, a standard feedforward neural baseline will be evaluated to establish a minimum neural performance threshold:
- Input: Vectorized token sequence.
- Embedding: Randomly initialized trainable embedding layer (e.g., dimension 128).
- Pooling: `GlobalAveragePooling1D` or Flatten.
- Dense: Hidden Dense layer with ReLU activation.
- Dropout: Regularization dropout (0.2–0.5).
- Output: Dense(6) with Softmax activation.


---
## 10. Recurrent Model Comparisons
> **Status: Planned / Not yet executed**

### 10.1 Comparative Recurrent Neural Architectures
To evaluate the relative efficacy of recurrent mechanisms, four distinct architectures will be benchmarked under strictly controlled training conditions:
1. **Simple RNN:** Vanilla recurrent neural network (prone to vanishing/exploding gradients on longer sequences).
2. **LSTM (Long Short-Term Memory):** Unidirectional gated recurrent architecture with cell state and forget gates.
3. **GRU (Gated Recurrent Unit):** Streamlined gated architecture with fewer parameters.
4. **Bi-LSTM (Bidirectional LSTM):** Bidirectional recurrent processing capturing both left-to-right and right-to-left contextual dependencies.

### 10.2 Fair Comparison Protocol
All comparative models will share:
- Identical train/val/test splits.
- Identical vocabulary size and sequence length.
- Identical embedding dimension and recurrent hidden unit count.
- Identical optimizer (`Adam`, learning rate $1 \times 10^{-3}$) and loss function (`sparse_categorical_crossentropy`).
- Early stopping monitored on validation loss.

| Architecture | Parameters | Val Loss | Val Macro F1 | Test Accuracy | Test Macro F1 | Status |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Simple RNN** | — | — | — | — | — | Not yet executed |
| **LSTM** | — | — | — | — | — | Not yet executed |
| **GRU** | — | — | — | — | — | Not yet executed |
| **Bi-LSTM** | — | — | — | — | — | Not yet executed |


---
## 11. Bi-LSTM Experiments
> **Status: Planned / Not yet executed**

### 11.1 Primary Architecture Specification
```
Input Text
    ↓
TextVectorization (vocab learned from Train only)
    ↓
Trainable Embedding Layer (d_embed = 128, trained from scratch)
    ↓
Bidirectional LSTM Layer (units = 64/128, return_sequences = False)
    ↓
Dropout Layer (rate = 0.3 - 0.5)
    ↓
Dense Layer (units = 64, activation = 'relu')
    ↓
Dropout Layer (rate = 0.3)
    ↓
Dense Output Layer (units = 6, activation = 'softmax')
```


---
## 12. Hyperparameter Experiments
> **Status: Planned / Not yet executed**

Planned controlled hyperparameter sweeps:
- **Sequence Length:** 64 vs 100 vs 128 tokens.
- **Vocabulary Size:** 10,000 vs 20,000 tokens.
- **Embedding Dimensions:** 64 vs 128 vs 256.
- **Recurrent Units:** 64 vs 128 units.
- **Dropout Regularization:** 0.2 vs 0.3 vs 0.5.
- **Batch Size & Learning Rate:** 32 / 64 batch sizes with Adam optimizer scheduling.


---
## 13. Ablation Studies
> **Status: Planned / Not yet executed**

Targeted ablation configurations to isolate performance drivers:
1. **Directionality Ablation:** Unidirectional LSTM vs Bidirectional LSTM.
2. **Regularization Ablation:** Bi-LSTM with dropout vs without dropout.
3. **Depth Ablation:** Single-layer Bi-LSTM vs Stacked 2-layer Bi-LSTM.


---
## 14. Final Evaluation
> **Status: Planned / Not yet executed**

Final evaluation will be conducted on the **untouched test set** only after architecture and hyperparameter selections are finalized.

Metrics to be reported:
- Overall Accuracy
- Macro Precision, Macro Recall, Macro F1
- Weighted F1
- Per-class classification report (Precision, Recall, F1, Support for each of the 6 classes)
- Multi-class Confusion Matrix heatmap


---
## 15. Error Analysis
> **Status: Planned / Not yet executed**

Systematic qualitative and quantitative analysis of test misclassifications:
- Confusion between `not_cyberbullying` and `other_cyberbullying`.
- Short vs long text error rates.
- Linguistic drivers: subtle sarcasm, implicit bias, out-of-vocabulary slang.


---
## 16. Results Summary
> **Status: Planned / Not yet executed**

A consolidated benchmark table will record every model variant, parameter count, training runtime, and validation/test metrics.

| Experiment ID | Model Architecture | Key Hyperparameters | Val Loss | Val Macro F1 | Test Accuracy | Test Macro F1 | Epochs | Training Time | Status |
| :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| EXP-00 | Neural Baseline | Dense / GlobalPool | — | — | — | — | — | — | Pending |
| EXP-01 | Simple RNN | 128 units | — | — | — | — | — | — | Pending |
| EXP-02 | LSTM | 128 units | — | — | — | — | — | — | Pending |
| EXP-03 | GRU | 128 units | — | — | — | — | — | — | Pending |
| EXP-04 | Bi-LSTM | 128 units (bidirectional) | — | — | — | — | — | — | Pending |


---
## 17. Conclusions
> **Status: Planned / Not yet executed**

Final takeaways, empirical conclusions, and research insights will be synthesized upon completion of all experimental stages.
